# 29 — SEC ESG Extractor

Pass in any public company ticker and get a CSRD-mapped ESG disclosure report pulled live from SEC EDGAR — no API key, no data subscription. The LLM maps each finding to a CSRD category and scores completeness.

In [ ]:
!pip install -q openai python-dotenv pydantic

In [ ]:
import os
os.environ['OPENAI_API_KEY'] = 'sk-...'  # replace

In [ ]:
from pydantic import BaseModel, Field

class ESGDisclosure(BaseModel):
    category: str = Field(description='CSRD reporting category.')
    topic: str = Field(description='Specific topic within the category.')
    disclosure_text: str = Field(description='Verbatim or close-paraphrase from the filing.')
    source_section: str = Field(description='10-K section where this was found.')
    completeness: str = Field(description='FULL, PARTIAL, or MINIMAL.')
    gaps: list[str] = Field(description='CSRD requirements missing for this topic.')

class ESGReport(BaseModel):
    company: str = Field(description='Company name.')
    ticker: str = Field(description='Stock ticker.')
    filing_year: int = Field(description='Fiscal year of the 10-K.')
    disclosures: list[ESGDisclosure] = Field(description='All ESG disclosures found, one per topic.')
    csrd_coverage_score: int = Field(description='CSRD coverage estimate 0-100.')
    strongest_areas: list[str] = Field(description='Up to 3 strongest CSRD categories.')
    critical_gaps: list[str] = Field(description='Up to 3 CSRD categories with no/minimal disclosure.')
    analyst_note: str = Field(description='2-3 sentence ESG disclosure assessment.')

In [ ]:
import json
import re
import urllib.request
from openai import OpenAI

EDGAR_TICKERS = 'https://www.sec.gov/files/company_tickers.json'
EDGAR_SUBMISSIONS = 'https://data.sec.gov/submissions/CIK{cik}.json'
HEADERS = {'User-Agent': 'agent-use-cases research@example.com'}

SYSTEM = (
    'You are an ESG analyst specialising in CSRD compliance. Given 10-K excerpts:\n'
    '1. Find all ESG disclosures and map each to a CSRD category\n'
    '2. Quote/paraphrase text and note the section\n'
    '3. Rate completeness (FULL/PARTIAL/MINIMAL) and list specific CSRD gaps\n'
    '4. Score CSRD coverage 0-100, name strongest areas and critical gaps\n'
    '5. Write a 2-3 sentence analyst note\n'
    'Cite sections. Do not invent disclosures.'
)

def _get(url):
    req = urllib.request.Request(url, headers=HEADERS)
    with urllib.request.urlopen(req, timeout=20) as r:
        return r.read()

def _cik(ticker):
    data = json.loads(_get(EDGAR_TICKERS))
    for entry in data.values():
        if entry.get('ticker', '').upper() == ticker.upper():
            return str(entry['cik_str']).zfill(10)
    raise ValueError(f'Ticker not found: {ticker}')

def _latest_10k(cik):
    data = json.loads(_get(EDGAR_SUBMISSIONS.format(cik=cik)))
    f = data['filings']['recent']
    for i, form in enumerate(f['form']):
        if form in ('10-K', '10-K/A'):
            acc = f['accessionNumber'][i].replace('-', '')
            return acc, int(f['filingDate'][i][:4])
    raise ValueError('No 10-K found')

def _filing_text(cik, accession, max_chars=40000):
    acc_d = f'{accession[:10]}-{accession[10:12]}-{accession[12:]}'
    url = f'https://www.sec.gov/Archives/edgar/data/{int(cik)}/{accession}/{acc_d}-index.json'
    try:
        idx = json.loads(_get(url))
        primary = next((f for f in idx.get('directory', {}).get('item', []) if f.get('name', '').endswith('.htm')), None)
        if primary:
            doc = _get(f'https://www.sec.gov/Archives/edgar/data/{int(cik)}/{accession}/{primary["name"]}').decode('utf-8', 'ignore')
            return re.sub(r'\s+', ' ', re.sub(r'<[^>]+>', ' ', doc))[:max_chars]
    except Exception:
        pass
    return ''

def _esg_sections(text, max_chars=30000):
    markers = ['environmental', 'climate', 'sustainability', 'governance', 'human capital', 'risk factor']
    chunks = []
    low = text.lower()
    for m in markers:
        i = low.find(m)
        if i != -1:
            chunks.append(text[max(0, i - 200):i + 3000])
    return (' ... '.join(chunks) or text)[:max_chars]

def analyse(ticker, company_name=None):
    cik = _cik(ticker)
    accession, year = _latest_10k(cik)
    raw = _filing_text(cik, accession)
    text = _esg_sections(raw) if raw else '(filing unavailable)'
    if not company_name:
        try:
            company_name = json.loads(_get(EDGAR_SUBMISSIONS.format(cik=cik))).get('name', ticker)
        except Exception:
            company_name = ticker
    prompt = f'Company: {company_name} ({ticker.upper()})\nFiling year: {year}\n\n{text}'
    client = OpenAI(api_key=os.environ['OPENAI_API_KEY'])
    r = client.beta.chat.completions.parse(
        model='gpt-4o-mini',
        messages=[{'role': 'system', 'content': SYSTEM}, {'role': 'user', 'content': prompt}],
        response_format=ESGReport
    )
    report = r.choices[0].message.parsed
    report.company = company_name
    report.ticker = ticker.upper()
    report.filing_year = year
    return report

In [ ]:
# Tech company with strong ESG disclosure
report = analyse('MSFT', 'Microsoft Corporation')
print(f'{report.company} ({report.ticker}) — {report.filing_year} 10-K')
print(f'CSRD Coverage: {report.csrd_coverage_score}/100 | Disclosures: {len(report.disclosures)}')
if report.strongest_areas:
    print(f'Strongest: {" | ".join(report.strongest_areas)}')
if report.critical_gaps:
    print(f'Gaps: {" | ".join(report.critical_gaps)}')
for d in report.disclosures[:4]:
    print(f'\n[{d.completeness}] {d.category} — {d.topic}')
    print(f'  {d.disclosure_text[:200]}...')
    if d.gaps:
        print(f'  Gaps: {d.gaps[0]}')
print(f'\nNote: {report.analyst_note}')

In [ ]:
# Energy major -- contrasting disclosure profile
report2 = analyse('XOM', 'Exxon Mobil Corporation')
print(f'{report2.company} ({report2.ticker}) — {report2.filing_year} 10-K')
print(f'CSRD Coverage: {report2.csrd_coverage_score}/100 | Disclosures: {len(report2.disclosures)}')
if report2.strongest_areas:
    print(f'Strongest: {" | ".join(report2.strongest_areas)}')
if report2.critical_gaps:
    print(f'Gaps: {" | ".join(report2.critical_gaps)}')
print(f'\nNote: {report2.analyst_note}')

## Exercise

Swap in a different ticker — try a mid-cap like `CROX` or `WBD` that may have thinner ESG disclosure. Compare `csrd_coverage_score` and `critical_gaps` against the large-cap examples. The same EDGAR pipeline runs regardless of company size.